<a href="https://colab.research.google.com/github/dlalswns0211/sparta_project_4_team16/blob/main/YJ/COLAB_DL_exp_FD001_BiLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ✈️ NASA C-MAPSS FD001 — 딥러닝 RUL 예측 모델링
# 전처리 조합 실험
---

**데이터셋:** NASA C-MAPSS FD001  
**목표:** 항공기 엔진 잔여 수명(RUL) 예측  
**모델:** BiLSTM  
**평가 지표:** RMSE, NASA Score  
**환경:** Google Colab T4 GPU  

---

## ⚠️ 주의사항
- `run_experiment(...)` 실행 코드는 주석 처리되어 있음
- 실험 실행 시 반드시 **세션 재시작 후 전체 실행** → 원하는 실험만 주석 해제
- AdamW 내부 상태(모멘텀)가 누적되므로 세션 재시작 없으면 실험 결과 오염 가능

In [ ]:
# from pyngrok import ngrok
# ngrok.kill()

# [0] 환경 설정

In [ ]:
# 설치 셀 (0번)
!pip install mlflow==2.13.0 pyngrok gunicorn -q --no-deps
import mlflow

/usr/local/lib/python3.12/dist-packages/mlflow/protos/service_pb2.py:11: UserWarning: google.protobuf.service module is deprecated. RPC implementations should provide code generator plugins which generate code specific to the RPC implementation. service.py will be removed in Jan 2025
  from google.protobuf import service as _service


In [ ]:
# 드라이브 마운트 먼저!
from google.colab import drive
drive.mount('/content/drive')
mlflow.set_tracking_uri("sqlite:////content/drive/MyDrive/sparta_project/mlflow.db")
!apt-get -qq install fonts-nanum

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mlflow.pytorch
import os
import json  # ← CSV 자동 저장 코드에 필요
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

import matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')
print(f'PyTorch 버전: {torch.__version__}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
사용 디바이스: cuda
PyTorch 버전: 2.10.0+cu128


# [1] MLflow + ngrok

In [ ]:
# import subprocess
# import time

# subprocess.run(['pkill', '-f', 'mlflow'], capture_output=True)
# subprocess.run(['pkill', '-f', 'ngrok'], capture_output=True)
# time.sleep(3)

# process = subprocess.Popen([
#     'mlflow', 'ui',
#     '--port', '5001',
#     '--host', '0.0.0.0',
#     '--backend-store-uri', 'sqlite:////content/drive/MyDrive/sparta_project/mlflow.db'
# ], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# time.sleep(10)

# # MLflow 서버 로그 확인
# stdout, stderr = process.communicate(timeout=1) if process.poll() else (b'', b'')
# print("에러:", stderr.decode()[:300] if stderr else "없음")

# # curl로 확인
# result = subprocess.run(['curl', 'http://localhost:5001'], capture_output=True, text=True)
# print("서버 응답:", result.stdout[:100] if result.stdout else "응답 없음")
# print("에러:", result.stderr[:100])

# from pyngrok import ngrok
# from google.colab import userdata
# ngrok.kill()
# time.sleep(2)
# ngrok.set_auth_token(userdata.get("NGROK_TOKEN"))
# public_url = ngrok.connect(5001)
# print(f"MLflow UI 주소: {public_url}")

In [ ]:
# import requests
# try:
#     r = requests.get("http://localhost:5001")
#     print("mlflow 서버 정상:", r.status_code)
# except:
#     print("mlflow 서버 안 떠있음")

# print(process.stderr.read().decode())

# [2] 데이터 로드 및 3D 복원

In [ ]:
BASE_DIR    = '/content/drive/MyDrive/sparta_project/preprocessed_FD001_exp'
DATA_DIR    = f'{BASE_DIR}/dl'
WINDOW_SIZE = 30  # WINDOW_SIZE W-1: 20, W-2: 10으로 변경 필수! 위에서 먼저 변경 후 실행

DL_FEATURES = ['s_2','s_3','s_4','s_7','s_8','s_9',
                's_11','s_12','s_13','s_14','s_15','s_17','s_20','s_21']

# B: 베이스라인 (cap=125 / σ=2 / win=30 / MinMax)
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win30_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win30_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win30_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap125_sig2_win30_test_RUL.csv')

# R-1: cap=120
subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap120_sig2_win30_dl_subtrain_long.csv')
valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap120_sig2_win30_dl_valid_long.csv')
test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap120_sig2_win30_dl_test_long.csv')
test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap120_sig2_win30_test_RUL.csv')

# R-2: cap=130
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap130_sig2_win30_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap130_sig2_win30_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap130_sig2_win30_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap130_sig2_win30_test_RUL.csv')

# G-1: σ=0
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig0_win30_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig0_win30_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig0_win30_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap125_sig0_win30_test_RUL.csv')

# G-2: σ=1
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig1_win30_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig1_win30_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig1_win30_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap125_sig1_win30_test_RUL.csv')

# G-3: σ=3
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig3_win30_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig3_win30_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig3_win30_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap125_sig3_win30_test_RUL.csv')

# N-1: Standard (WINDOW_SIZE=30 유지)
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win30_standard_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win30_standard_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win30_standard_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap125_sig2_win30_standard_test_RUL.csv')

# W-1: window=20 (WINDOW_SIZE=20 변경 필수!)
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win20_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win20_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win20_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap125_sig2_win20_test_RUL.csv')

# W-2: window=10 (WINDOW_SIZE=10 변경 필수!)
# subtrain_df = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win10_dl_subtrain_long.csv')
# valid_df    = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win10_dl_valid_long.csv')
# test_df     = pd.read_csv(f'{DATA_DIR}/FD001_cap125_sig2_win10_dl_test_long.csv')
# test_rul_df = pd.read_csv(f'{BASE_DIR}/FD001_cap125_sig2_win10_test_RUL.csv')

def long_to_3d(df):
    grp = df.groupby('sample_id')
    X = np.stack([grp.get_group(i)[DL_FEATURES].values
                  for i in range(df['sample_id'].nunique())])
    y = df.groupby('sample_id')['RUL'].last().values
    return X, y

X_train, y_train = long_to_3d(subtrain_df)
X_valid, y_valid = long_to_3d(valid_df)
X_test,  _       = long_to_3d(test_df)
y_test           = test_rul_df['RUL'].values

print(f'X_train: {X_train.shape} / y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape} / y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}  / y_test:  {y_test.shape}')

X_train: (14241, 30, 14) / y_train: (14241,)
X_valid: (3490, 30, 14) / y_valid: (3490,)
X_test:  (100, 30, 14)  / y_test:  (100,)


# [3] Dataset / DataLoader

In [ ]:
class RULDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 64

train_dataset = RULDataset(X_train, y_train)
valid_dataset = RULDataset(X_valid, y_valid)
test_dataset  = RULDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# [4] 평가 함수

In [ ]:
def nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    score = np.sum(np.where(diff < 0,
                            np.exp(-diff / 13) - 1,
                            np.exp(diff / 10) - 1))
    return score

def evaluate(model, loader, device):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            out = model(X_batch).cpu().numpy()
            preds.extend(out)
            targets.extend(y_batch.numpy())
    preds   = np.array(preds)
    targets = np.array(targets)
    rmse  = np.sqrt(np.mean((preds - targets) ** 2))
    score = nasa_score(targets, preds)
    return rmse, score, preds, targets

# [5] 학습 함수

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X_batch)
    return total_loss / len(loader.dataset)

# [6] 시각화 함수

In [ ]:
def plot_results(exp_name, model_name, train_loss_list, val_rmse_list,
                 test_preds, test_targets):

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f'{model_name} - {exp_name} (FD001)', fontsize=14, fontweight='bold')

    ax = axes[0]
    ax.plot(train_loss_list, label='Train Loss')
    ax.plot(val_rmse_list,   label='Val RMSE')
    ax.set_title('학습 곡선')
    ax.set_xlabel('Epoch')
    ax.legend()

    ax = axes[1]
    ax.scatter(test_targets, test_preds, alpha=0.5, s=20)
    max_val = max(test_targets.max(), test_preds.max())
    ax.plot([0, max_val], [0, max_val], 'r--', label='이상적 예측')
    ax.set_title('예측값 vs 실제값 (Test)')
    ax.set_xlabel('실제 RUL')
    ax.set_ylabel('예측 RUL')
    ax.legend()

    ax = axes[2]
    residuals = test_preds - test_targets
    ax.hist(residuals, bins=20, edgecolor='black')
    ax.axvline(0, color='red', linestyle='--', label='잔차=0')
    ax.set_title('잔차 분포 (Test)')
    ax.set_xlabel('예측 - 실제')
    ax.set_ylabel('빈도')
    ax.legend()

    plt.tight_layout()
    plt.show()

# [7] 모델 정의

In [ ]:
class BiLSTMModel(nn.Module):
    def __init__(self, input_size=14, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.bilstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out).squeeze(1)

# [8] 실험 함수


In [ ]:
EPOCHS   = 100
LR       = 1e-3
PATIENCE = 20

def run_experiment(exp_name, model_class, hidden_size=64, num_layers=2, dropout=0.2):
    exp_name_full = f"{model_class.__name__}_FD001"
    mlflow.set_experiment(exp_name_full)

    with mlflow.start_run(run_name=exp_name):
        mlflow.log_params({
            "model":       model_class.__name__,
            "hidden_size": hidden_size,
            "num_layers":  num_layers,
            "dropout":     dropout,
            "batch_size":  BATCH_SIZE,
            "lr":          LR,
            "epochs":      EPOCHS,
            "dataset":     "FD001",
            "optimizer":   "Adam",
            "loss":        "MSELoss",
            "window_size": WINDOW_SIZE
        })

        model = model_class(
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.MSELoss()
        # criterion = nn.HuberLoss(delta=10.0)  # ← 추후 최적조합 추가실험 시 비교용
        best_val_rmse = float('inf')
        patience_cnt  = 0
        train_loss_list = []
        val_rmse_list   = []

        for epoch in range(1, EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_rmse, val_score, _, _ = evaluate(model, valid_loader, device)

            train_loss_list.append(train_loss)
            val_rmse_list.append(val_rmse)

            mlflow.log_metrics({
                "train_loss": train_loss,
                "val_rmse":   val_rmse,
                "val_score":  val_score
            }, step=epoch)

            if val_rmse < best_val_rmse:
                best_val_rmse = val_rmse
                best_state    = model.state_dict().copy()
                patience_cnt  = 0
            else:
                patience_cnt += 1

            if epoch % 10 == 0:
                print(f"Epoch {epoch:3d} | train_loss: {train_loss:.4f} | val_rmse: {val_rmse:.4f}")

            if patience_cnt >= PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

        model.load_state_dict(best_state)
        test_rmse, test_score, test_preds, test_targets = evaluate(model, test_loader, device)

        mlflow.log_metrics({"test_rmse": test_rmse, "test_score": test_score})
        mlflow.pytorch.log_model(model, "model")

        # ── CSV 자동 저장 ──────────────────────────────────────────
        BEST_DIR = '/content/drive/MyDrive/sparta_project/best_results'
        os.makedirs(BEST_DIR, exist_ok=True)

        MODEL_NAME = model_class.__name__
        best_meta_path = f'{BEST_DIR}/FD001_{MODEL_NAME}_best.json'

        if os.path.exists(best_meta_path):
            with open(best_meta_path) as f:
                prev_best = json.load(f)
            prev_rmse = prev_best['test_rmse']
        else:
            prev_rmse = float('inf')

        if test_rmse < prev_rmse:
            pd.DataFrame({
                'engine_id':     range(len(test_preds)),
                'actual_rul':    test_targets,
                'predicted_rul': test_preds
            }).to_csv(f'{BEST_DIR}/FD001_{MODEL_NAME}_pred.csv', index=False)

            with open(best_meta_path, 'w') as f:
                json.dump({
                    'exp_name':   exp_name,
                    'model':      MODEL_NAME,
                    'dataset':    'FD001',
                    'test_rmse':  float(test_rmse),
                    'nasa_score': float(test_score)
                }, f, indent=2)

            print(f"🏆 최고 성능 갱신! RMSE: {test_rmse:.4f} → 드라이브 저장 완료")
        else:
            print(f"📊 현재 RMSE: {test_rmse:.4f} | 기존 최고: {prev_rmse:.4f} → 저장 안함")
        # ───────────────────────────────────────────────────────────

        print(f"\n{'='*40}")
        print(f"[{exp_name}] 최종 결과")
        print(f"  Test RMSE  : {test_rmse:.4f}")
        print(f"  NASA Score : {test_score:.2f}")
        print(f"{'='*40}")

        plot_results(
            exp_name        = exp_name,
            model_name      = model_class.__name__,
            train_loss_list = train_loss_list,
            val_rmse_list   = val_rmse_list,
            test_preds      = test_preds,
            test_targets    = test_targets
        )

        return test_rmse, test_score

# [9] 실험 실행 셀


In [ ]:
# ── 전처리 실험 (Adam, 베이스라인 파라미터 고정) ──────────
# 데이터 로드 셀에서 CSV 경로 주석 변경 + WINDOW_SIZE 확인 후 세션 재시작 → 전체 실행 → 주석 해제

# B: 베이스라인 (cap=125 / σ=2 / win=30 / MinMax)
# run_experiment("B", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# R-1: cap=120
# run_experiment("R-1", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# R-2: cap=130
# run_experiment("R-2", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# G-1: σ=0
# run_experiment("G-1", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# G-2: σ=1
# run_experiment("G-2", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# G-3: σ=3
# run_experiment("G-3", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# N-1: Standard (데이터 로드 셀 경로 + WINDOW_SIZE 확인)
# run_experiment("N-1", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# W-1: window=20 (WINDOW_SIZE=20 변경 필수!)
# run_experiment("W-1", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# W-2: window=10 (WINDOW_SIZE=10 변경 필수!)
# run_experiment("W-2", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

나중에 최종 실험 때 필요할 수도 있을 것 같아서 놔둠.

In [ ]:
# 베이스라인
# run_experiment("B-1", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.2)

# 하이퍼파라미터 실험
# H-1: 층수 3
# run_experiment("H-1", BiLSTMModel, hidden_size=64, num_layers=3, dropout=0.2)

# H-2: 층수 4
# run_experiment("H-2", BiLSTMModel, hidden_size=64, num_layers=4, dropout=0.2)

# H-3: 은닉층 128
# run_experiment("H-3", BiLSTMModel, hidden_size=128, num_layers=2, dropout=0.2)

# H-4: 은닉층 256
# run_experiment("H-4", BiLSTMModel, hidden_size=256, num_layers=2, dropout=0.2)

# H-5: 드롭아웃 0.3
# run_experiment("H-5", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.3)

# H-6: 드롭아웃 0.5
# run_experiment("H-6", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.5)

# 최종 하이퍼 파라미터 + 전처리 조합 실험

In [ ]:
# → 두 요소를 결합했을 때 시너지 효과가 있는지 확인하기 위한 실험
# 모든 실험 완료 후 성능 결과 확인 후 코드 변경 예정
# BiLSTM: G-1(σ=0, 14.13)이 베이스라인(15.30)보다 확실히 개선됐으니 G-1 + H-6(D0.5) 조합은 해볼 만함/ 가우시안 0실험 + 드롭아웃 0.5 또는 민준님께 가우시안 0 실험을 옵튜나로 실험 권유

# run_experiment("F-1", BiLSTMModel, hidden_size=64, num_layers=2, dropout=0.5)